# TUGAS MANDIRI — PERTEMUAN 4
### Pemrosesan Big Data dengan Apache Spark & PySpark DataFrame

| Informasi | Detail |
|---|---|
| **Mata Kuliah** | Praktikum Big Data |
| **Pertemuan** | 4 |
| **Topik** | Instalasi Apache Spark & Operasi Dasar PySpark DataFrame |
| **Nama Mahasiswa** | *(Isi Nama Anda)* |
| **NPM** | *(Isi NPM Anda)* |

---
## Skenario Tugas
Memindahkan alur pemrosesan data e-commerce dari Pandas ke **PySpark DataFrame** untuk menangani dataset skala besar yang disimpan langsung di **HDFS**.


### Persiapan Dataset & Upload ke HDFS


In [ ]:
import numpy as np
import pandas as pd
import os

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Unggah ke HDFS jika Hadoop aktif
!hdfs dfs -mkdir -p /user/mahasiswa/tugas4 2>/dev/null || true
!hdfs dfs -put -f transaksi_september_2026.csv /user/mahasiswa/tugas4/ 2>/dev/null || true


### Bagian A: Membaca & Eksplorasi Awal dengan PySpark


In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, count, sum as spark_sum, desc

spark = SparkSession.builder \
    .appName("Tugas4_PySpark") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# Membaca data (mencoba dari HDFS, fallback ke file lokal)
path_hdfs = "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv"
path_local = "transaksi_september_2026.csv"

try:
    df_spark = spark.read.csv(path_hdfs, header=True, inferSchema=True)
    print("Berhasil membaca dari HDFS!")
except Exception:
    df_spark = spark.read.csv(path_local, header=True, inferSchema=True)
    print("Membaca dari file lokal.")

print("Skema Data:")
df_spark.printSchema()

print(f"Total Baris: {df_spark.count()}")
print("\n10 Baris Pertama:")
df_spark.show(10)


### Bagian B: Menangani Data Kosong pada Kolom `rating`


In [ ]:
# 1. Menghitung jumlah missing value pada kolom rating
null_count = df_spark.filter(col("rating").isNull()).count()
print(f"Jumlah baris dengan rating kosong: {null_count}")

# 2. Penanganan: Mengisi nilai null dengan angka 0
# Alasan: Rating kosong menandakan pembeli belum/tidak memberikan ulasan, bukan rating rendah (1-5).
# Memberikan nilai 0 membedakan secara jelas transaksi tanpa ulasan dengan transaksi berulasan.
df_clean = df_spark.na.fill({"rating": 0})

print("Verifikasi data kosong setelah penanganan:")
print("Null rating tersisa:", df_clean.filter(col("rating").isNull()).count())


### Bagian C: Transformasi Data (`total_pendapatan` & `tier_transaksi`)


In [ ]:
# Menambahkan kolom total_pendapatan dan tier_transaksi
df_transformed = df_clean.withColumn(
    "total_pendapatan", col("unit_terjual") * col("harga_satuan")
).withColumn(
    "tier_transaksi",
    when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil")
)

df_transformed.select("order_id", "kategori", "unit_terjual", "harga_satuan", "total_pendapatan", "tier_transaksi").show(10)


### Bagian D: Analisis dengan GroupBy


In [ ]:
# 1. Kategori dengan total pendapatan tertinggi
print("1. Kategori dengan Total Pendapatan Tertinggi:")
top_kategori = df_transformed.groupBy("kategori").agg(
    spark_sum("total_pendapatan").alias("pendapatan_kategori")
).orderBy(desc("pendapatan_kategori"))
top_kategori.show(1)

# 2. Kota dengan jumlah transaksi tier 'Besar' terbanyak
print("2. Kota dengan Transaksi Tier 'Besar' Terbanyak:")
top_kota_besar = df_transformed.filter(col("tier_transaksi") == "Besar").groupBy("kota").agg(
    count("order_id").alias("jumlah_transaksi_besar")
).orderBy(desc("jumlah_transaksi_besar"))
top_kota_besar.show(1)

# 3. Rata-rata rating untuk masing-masing metode pembayaran
print("3. Rata-rata Rating per Metode Pembayaran:")
avg_rating_metode = df_transformed.groupBy("metode_pembayaran").agg(
    avg("rating").alias("rata_rata_rating")
).orderBy(desc("rata_rata_rating"))
avg_rating_metode.show()


### Bagian E: Menyimpan Hasil ke HDFS & Penjelasan Partisi


In [ ]:
# Menyimpan hasil ke direktori partisi CSV
output_path = "output_transaksi_september"
df_transformed.write.mode("overwrite").option("header", "true").csv(output_path)

print(f"Hasil olahan berhasil disimpan ke folder '{output_path}'.")
!ls -la {output_path}


#### Penjelasan Mengapa Spark Menyimpan Hasil Sebagai Berkas Partisi:
Apache Spark adalah mesin komputasi terdistribusi. Saat memproses RDD/DataFrame, data dibagi menjadi beberapa partisi yang diproses secara paralel oleh core CPU / node pekerja (executors). Saat instruksi `write` dijalankan, setiap executor menuliskan partisi data yang ditanganinya ke berkas terpisah (seperti `part-00000`, `part-00001`). Pendekatan ini memungkinkan operasi I/O berjalan paralel dan sangat cepat tanpa bottleneck pada satu file tunggal.
